# Vitessce Widget Tutorial

# Example usage of Neuroglancer precomputed segmentations with multiple point annotation layers

This notebook demonstrates `ObsSegmentationsNgPrecomputedWrapper`, `ObsPointsNgAnnotationsWrapper`, and `ObsLocationsNgAnnotationsWrapper`, which wrap Neuroglancer precomputed segmentation/mesh data and point-annotation data (e.g. as produced by the [tissue-map-tools](https://github.com/hms-dbmi/tissue-map-tools) library) for use with the `neuroglancer` and `layerControllerBeta` views.

Two kinds of point annotations are shown alongside the cell segmentation meshes:
- `ObsPointsNgAnnotationsWrapper` (`obsPoints.ng-annotations`): protein spots, rendered as their own point layer.
- `ObsLocationsNgAnnotationsWrapper` (`obsLocations.ng-annotations`): cell centroids. These are rendered implicitly together with the segmentation layer of the same `obsType`, using its visual properties (visibility, opacity, color encoding, etc.).

In [ ]:
from vitessce import (
    VitessceConfig,
    CoordinationLevel as CL,
    get_initial_coordination_scope_prefix,
    ObsSegmentationsNgPrecomputedWrapper,
    ObsPointsNgAnnotationsWrapper,
    ObsLocationsNgAnnotationsWrapper,
    CsvWrapper,
)

## 1. Configure Vitessce

In [ ]:
base_url = "https://data-2.vitessce.io/data/sorger/tissue-map-tools-output-tab"

vc = VitessceConfig(schema_version="1.0.17", name="Sorger MIS")
dataset = vc.add_dataset("My dataset")

# A Neuroglancer precomputed segmentation + meshes directory.
# fileUid here must match the value used below in link_views_by_dict's
# segmentationLayer coordination.
dataset.add_object(ObsSegmentationsNgPrecomputedWrapper(
    data_url=f"{base_url}/MIS_combined_precomputed",
    coordination_values={"fileUid": "mis-meshes", "obsType": "cell"},
))

# An obsSets-type file can be used for segments to be dynamically
# selected/colored -- obsType here must match the segmentationChannel's
# obsType coordination value set below.
dataset.add_object(CsvWrapper(
    csv_url=f"{base_url}/MIS_combined_metadata_V2.csv",
    data_type="obsSets",
    coordination_values={"obsType": "cell"},
    options={
        "obsIndex": "CellID",
        "obsSets": [{"name": "Cell Types", "column": "phenotype"}],
    },
))

# Protein spots, feature-colorable (gene/protein selection).
# fileUid here must match the value used below in the pointLayer coordination.
dataset.add_object(ObsPointsNgAnnotationsWrapper(
    data_url=f"{base_url}/MIS_GZMB_spotmask2_matched_spot_coordinates_with_protein_annotations_precomputed/proteins",
    coordination_values={"fileUid": "mis-proteins", "obsType": "point", "featureType": "gene"},
    options={
        "projectionAnnotationSpacing": 2.4544585683772735,
        "featureIndexProp": "protein",
        "transform": {
            "matrix": [
                [1000, 0, 0, 0],
                [0, 1000, 0, 0],
                [0, 0, 1000, 0],
            ],
            "outputDimensions": {"x": [1e-9, "m"], "y": [1e-9, "m"], "z": [1e-9, "m"]},
        },
    },
))

# Cell centroids: rendered implicitly with the segmentation layer of the same obsType,
# using its visual properties (visibility, opacity, color encoding, etc.).
dataset.add_object(ObsLocationsNgAnnotationsWrapper(
    data_url=f"{base_url}/MIS_combined_annotations_corrected_precomputed/cells",
    coordination_values={"obsType": "cell"},
    options={
        "projectionAnnotationSpacing": 0.5,
        "transform": {
            "matrix": [
                [7148.09960682, 0, 0, 0],
                [0, 7148.09960682, 0, 0],
                [0, 0, 3803.92156863, 0],
            ],
            "outputDimensions": {"x": [1e-9, "m"], "y": [1e-9, "m"], "z": [1e-9, "m"]},
        },
    },
))

In [ ]:
ng_view = vc.add_view("neuroglancer", dataset=dataset).set_props(
    initialNgCameraState={
        "position": [2727000, 1377000, 97000],  # centre of the segmentation volume
        "projectionScale": 2700000,
    },
    meshLoadProjectionScaleThreshold=300000,
    showAxisLines=False,
)
lc_view = vc.add_view("layerControllerBeta", dataset=dataset)
obs_sets_view = vc.add_view("obsSets", dataset=dataset)
vc.layout(ng_view | (lc_view / obs_sets_view))

## 2. Coordinate the views

Four separate `link_views_by_dict` calls are needed:
- A plain (non-meta) link for shared camera position/rotation.
- A multi-level (meta) link for the segmentation layer + channel, mirroring the shape
  `obsSegmentations.ng-precomputed` files require to resolve correctly. The cell centroids
  from the `obsLocations.ng-annotations` file inherit this layer's visual properties.
- A multi-level (meta) link for the protein point layer.
- A link of the point layer's `obsType` and `featureType`, so that views such as `featureList`
  can be scoped to the protein point layer.

In [ ]:
vc.link_views_by_dict([ng_view, lc_view], {
    "spatialZoom": 0,
    "spatialTargetT": 0,
    "spatialTargetX": 0,
    "spatialTargetY": 0,
    "spatialTargetZ": 0,
    "spatialRotationX": 0,
    "spatialRotationY": 0,
    "spatialRotationZ": 0,
    "spatialRotationOrbit": 0,
}, meta=False)

vc.link_views_by_dict([ng_view, lc_view], {
    "segmentationLayer": CL([{
        "fileUid": "mis-meshes",
        "spatialLayerOpacity": 1,
        "spatialLayerVisible": True,
        "segmentationChannel": CL([{
            "obsType": "cell",
            "spatialChannelVisible": True,
            "spatialChannelOpacity": 0.6,
            "obsColorEncoding": "cellSetSelection",
        }]),
    }]),
}, scope_prefix=get_initial_coordination_scope_prefix("A", "obsSegmentations"))

vc.link_views_by_dict([ng_view, lc_view], {
    "pointLayer": CL([{
        "fileUid": "mis-proteins",
        "obsType": "point",
        "featureType": "gene",
        "spatialLayerOpacity": 1,
        "spatialLayerVisible": True,
        "spatialLayerLabel": "Protein",
        "spatialPointStrokeWidth": 0.2,
        "obsColorEncoding": "spatialLayerColor",
    }]),
}, scope_prefix=get_initial_coordination_scope_prefix("A", "obsPoints"))

vc.link_views_by_dict([ng_view, lc_view], {
    "obsType": "point",
    "featureType": "gene",
}, scope_prefix=get_initial_coordination_scope_prefix("A", "obsPoints"))

## 3. Create the Vitessce widget

In [ ]:
vw = vc.widget()
vw